# DeepSets Extrapolation Testing

This notebook evaluates trained DeepSets models from the extrapolation experiments on unseen distances (d=9, d=11, d=13).

**Workflow:**
1. Load trained models from different split experiments
2. Generate/load test datasets for d=9, d=11, and d=13
3. Evaluate accuracy across different physical error rates
4. Compare extrapolation performance across split configurations
5. Compare with GraphSAGE and MWPM baselines

**Key Questions:**
- Can DeepSets models trained on d=3,5,7 generalize to d=9,11,13?
- Does the training data split (d=3/d=5/d=7 ratio) affect extrapolation?
- How does DeepSets compare to GraphSAGE for extrapolation?

## Imports

In [ ]:
import sys
import json
import random
import time
from pathlib import Path
from datetime import datetime

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/Research/QEC/quantum-error-correction/quantum-error-correction/code')
else:
    BASE_PATH = Path('../..')  # code/deepsets/extrapolation -> code/

sys.path.insert(0, str(BASE_PATH))

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm

# Import from benchmark_models.py
from benchmark_models import (
    SurfaceCodeSampler,
    DeepSetsModel,
    DeepSets,
    FlatDatasetCache,
    ler_mwpm,
)

# Set up paths
EXTRAPOLATION_DIR = BASE_PATH / "deepsets" / "extrapolation"
RESULTS_DIR = EXTRAPOLATION_DIR / "results"
MODELS_DIR = EXTRAPOLATION_DIR / "models"
PLOTS_DIR = EXTRAPOLATION_DIR / "plots"

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print(f"\nPaths:")
print(f"  BASE_PATH: {BASE_PATH}")
print(f"  MODELS_DIR: {MODELS_DIR}")
print(f"  RESULTS_DIR: {RESULTS_DIR}")
print(f"  PLOTS_DIR: {PLOTS_DIR}")

## Configuration

In [ ]:
# =============================================================================
# TESTING CONFIGURATION
# =============================================================================

# Distances to test extrapolation on
TEST_DISTANCES = [9, 11, 13]

# Physical error rates for testing
P_VALUES = [0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008]

# Number of samples per distance per error rate
SAMPLES_PER_P = 10000

# Batch size for inference
BATCH_SIZE = 256

# Random seed
SEED = 42

print(f"Testing Configuration:")
print(f"  Test distances: {TEST_DISTANCES}")
print(f"  P values: {P_VALUES}")
print(f"  Samples per (distance, p): {SAMPLES_PER_P:,}")
print(f"  Total test samples per distance: {SAMPLES_PER_P * len(P_VALUES):,}")

In [ ]:
# =============================================================================
# SPLIT EXPERIMENTS (must match training.ipynb)
# =============================================================================

SPLIT_EXPERIMENTS = {
    'baseline_33_33_33': {'d3': 0.33, 'd5': 0.33, 'd7': 0.34, 'hypothesis': 'Reference baseline'},
    'no_d3_50_50': {'d3': 0.00, 'd5': 0.50, 'd7': 0.50, 'hypothesis': 'Is d=3 useless?'},
    'no_d3_30_70': {'d3': 0.00, 'd5': 0.30, 'd7': 0.70, 'hypothesis': 'Does more d=7 help?'},
    'tiny_d3_10_40_50': {'d3': 0.10, 'd5': 0.40, 'd7': 0.50, 'hypothesis': 'Does tiny d=3 help?'},
    'reversed_50_30_20': {'d3': 0.50, 'd5': 0.30, 'd7': 0.20, 'hypothesis': 'Sanity check'},
}

print("Split Experiments:")
for name in SPLIT_EXPERIMENTS:
    print(f"  - {name}")

## Helper Functions

In [ ]:
def list_available_models():
    """
    List all trained models in the models directory.

    Returns:
        List of model names (without .pt extension)
    """
    models = []
    if MODELS_DIR.exists():
        for model_file in MODELS_DIR.glob("*.pt"):
            models.append(model_file.stem)
    return sorted(models)

print("Available trained models:")
available_models = list_available_models()
if available_models:
    for m in available_models:
        print(f"  - {m}")
else:
    print("  No models found. Run training.ipynb first.")

In [ ]:
def load_trained_model(split_name: str):
    """
    Load a trained model from the extrapolation models directory.

    Args:
        split_name: Name of the split experiment

    Returns:
        Tuple of (DeepSets model wrapper, metadata dict)
    """
    model_path = MODELS_DIR / f"{split_name}.pt"

    if not model_path.exists():
        raise FileNotFoundError(f"Model not found: {model_path}")

    # Load saved data
    save_dict = torch.load(model_path, map_location=device, weights_only=False)
    config = save_dict['config']

    # Create model with saved config
    model = DeepSets(
        nickname=f"extrapolation_{split_name}",
        phi_hidden=tuple(config['phi_hidden']) if isinstance(config['phi_hidden'], list) else config['phi_hidden'],
        rho_hidden=tuple(config['rho_hidden']) if isinstance(config['rho_hidden'], list) else config['rho_hidden'],
        pool=config['pool'],
        dropout=config.get('dropout', 0.0),
        device=device,
        base_path=BASE_PATH,
    )

    # Load weights
    model.model.load_state_dict(save_dict['state_dict'])
    model.model.eval()

    metadata = {
        'split_name': save_dict.get('split_name', split_name),
        'split_config': save_dict.get('split_config', {}),
        'hyperparams': save_dict.get('hyperparams', {}),
        'metrics': save_dict.get('metrics', {}),
        'timestamp': save_dict.get('timestamp', ''),
    }

    print(f"Loaded model: {split_name}")
    print(f"  Config: phi_hidden={config['phi_hidden']}, rho_hidden={config['rho_hidden']}, pool={config['pool']}")
    if metadata['metrics']:
        print(f"  Training val_acc: {metadata['metrics'].get('val_accuracy', 0)*100:.2f}%")

    return model, metadata

In [ ]:
def generate_test_dataset(d: int, p: float, n_samples: int):
    """
    Generate test dataset for a specific distance and error rate.

    Args:
        d: Code distance
        p: Physical error rate
        n_samples: Number of samples

    Returns:
        Tuple of (detections, labels) tensors
    """
    sampler = SurfaceCodeSampler(p=p, device=device)

    # Generate samples
    detections, labels = sampler.sample(
        d=d,
        num_samples=n_samples,
        p_values=[p],
        p_weights=[1.0]
    )

    return detections, labels

In [ ]:
def load_or_generate_test_data(d: int, n_samples_per_p: int, p_values: list, use_cache: bool = True):
    """
    Load test data from cache or generate it.

    Args:
        d: Code distance
        n_samples_per_p: Samples per error rate
        p_values: List of error rates
        use_cache: Whether to try loading from cache

    Returns:
        Dict mapping p -> (detections, labels)
    """
    cache_name = f"d{d}_extrapolation_test"

    # Try to load from cache
    if use_cache:
        cache = FlatDatasetCache(base_path=BASE_PATH, device=device)
        try:
            cache.load(cache_name, verbose=True)
            total_needed = n_samples_per_p * len(p_values)
            if len(cache) >= total_needed:
                print(f"Using cached dataset: {cache_name}")
                detections, labels = cache.get_data(shuffle=False)
                # Split by p_value (assuming equal distribution in cache)
                samples_per_p = len(labels) // len(p_values)
                return {p: (detections[i*samples_per_p:(i+1)*samples_per_p],
                           labels[i*samples_per_p:(i+1)*samples_per_p])
                        for i, p in enumerate(p_values)}
        except FileNotFoundError:
            print(f"Cache not found: {cache_name}. Generating...")

    # Generate data for each p value
    test_data = {}
    print(f"\nGenerating test data for d={d}...")

    for p in tqdm(p_values, desc=f"d={d}"):
        detections, labels = generate_test_dataset(d, p, n_samples_per_p)
        test_data[p] = (detections, labels)

    return test_data

In [ ]:
def evaluate_model_on_data(model, detections, labels, batch_size=256):
    """
    Evaluate DeepSets model accuracy on test data.

    DeepSets can handle variable-size inputs, so no padding is needed.

    Args:
        model: DeepSets model wrapper
        detections: Tensor of shape [n_samples, num_detectors]
        labels: Tensor of shape [n_samples]
        batch_size: Batch size for inference

    Returns:
        Accuracy as float
    """
    model.model.eval()

    correct = 0
    total = len(labels)

    with torch.no_grad():
        for i in range(0, total, batch_size):
            X = detections[i:i+batch_size].unsqueeze(-1)  # [batch, N, 1]
            y = labels[i:i+batch_size]
            pred = model.model(X)
            correct += ((pred.squeeze() > 0.5).float() == y).sum().item()

    return correct / total if total > 0 else 0.0

In [ ]:
def compute_mwpm_baseline(d: int, p_values: list, n_shots: int = 10000):
    """
    Compute MWPM decoder logical error rate for comparison.

    Args:
        d: Code distance
        p_values: List of error rates
        n_shots: Number of shots per error rate

    Returns:
        Dict mapping p -> logical error rate
    """
    mwpm_ler = {}
    print(f"\nComputing MWPM baseline for d={d}...")

    for p in tqdm(p_values, desc=f"MWPM d={d}"):
        ler = ler_mwpm(p, d, num_shots=n_shots)
        mwpm_ler[p] = ler

    return mwpm_ler

## Load Models and Test Data

In [ ]:
# Select which models to evaluate
# Set to None to evaluate all available models, or specify a list
MODELS_TO_EVALUATE = None  # e.g., ['baseline_33_33_33', 'no_d3_50_50']

if MODELS_TO_EVALUATE is None:
    models_to_evaluate = list_available_models()
else:
    models_to_evaluate = MODELS_TO_EVALUATE

if not models_to_evaluate:
    print("WARNING: No models to evaluate. Run training.ipynb first.")
else:
    print(f"Models to evaluate: {models_to_evaluate}")

In [ ]:
# Load all models
loaded_models = {}

for split_name in models_to_evaluate:
    try:
        model, metadata = load_trained_model(split_name)
        loaded_models[split_name] = {'model': model, 'metadata': metadata}
    except FileNotFoundError as e:
        print(f"Skipping {split_name}: {e}")

print(f"\nLoaded {len(loaded_models)} models")

In [ ]:
# Generate or load test data for extrapolation distances
test_data = {}

for d in TEST_DISTANCES:
    print(f"\n{'='*50}")
    print(f"Loading/generating test data for d={d}")
    print(f"{'='*50}")
    test_data[d] = load_or_generate_test_data(d, SAMPLES_PER_P, P_VALUES, use_cache=True)

print(f"\nTest data loaded for distances: {list(test_data.keys())}")

## Evaluate Extrapolation

In [ ]:
# Evaluate all models on all test data
results = []

print(f"\n{'='*70}")
print("EVALUATING EXTRAPOLATION PERFORMANCE")
print(f"{'='*70}")

for split_name, model_info in loaded_models.items():
    model = model_info['model']
    print(f"\nEvaluating: {split_name}")

    for d in TEST_DISTANCES:
        print(f"  Distance d={d}:")
        for p in tqdm(P_VALUES, desc=f"    p values", leave=False):
            detections, labels = test_data[d][p]
            accuracy = evaluate_model_on_data(model, detections, labels, BATCH_SIZE)

            results.append({
                'split_name': split_name,
                'distance': d,
                'p_value': p,
                'accuracy': accuracy,
                'error_rate': 1 - accuracy,  # Model's logical error rate
            })

        # Print summary for this distance
        d_results = [r for r in results if r['split_name'] == split_name and r['distance'] == d]
        avg_acc = np.mean([r['accuracy'] for r in d_results])
        print(f"    Average accuracy: {avg_acc*100:.2f}%")

In [ ]:
# Convert to DataFrame
df_results = pd.DataFrame(results)

print("\nResults DataFrame:")
print(df_results.head(20))

## Compute MWPM Baseline

In [ ]:
# Compute MWPM baseline for comparison
mwpm_results = {}

for d in TEST_DISTANCES:
    mwpm_results[d] = compute_mwpm_baseline(d, P_VALUES, n_shots=SAMPLES_PER_P)

print("\nMWPM Baseline Results:")
for d, ler_dict in mwpm_results.items():
    avg_acc = 1 - np.mean(list(ler_dict.values()))
    print(f"  d={d}: average accuracy = {avg_acc*100:.2f}%")

## Results Summary

In [ ]:
# Create summary table by split and distance
summary_data = []

for split_name in loaded_models.keys():
    for d in TEST_DISTANCES:
        split_d_results = df_results[(df_results['split_name'] == split_name) & (df_results['distance'] == d)]
        avg_acc = split_d_results['accuracy'].mean()
        avg_ler = split_d_results['error_rate'].mean()

        # MWPM comparison
        mwpm_avg_ler = np.mean(list(mwpm_results[d].values()))
        mwpm_avg_acc = 1 - mwpm_avg_ler

        summary_data.append({
            'Split': split_name,
            'Distance': d,
            'DeepSets Acc': avg_acc * 100,
            'MWPM Acc': mwpm_avg_acc * 100,
            'Improvement': (avg_acc - mwpm_avg_acc) * 100,
        })

df_summary = pd.DataFrame(summary_data)
print("\nExtrapolation Results Summary:")
print(df_summary.to_string(index=False))

In [ ]:
# Save results
results_path = RESULTS_DIR / "extrapolation_results.json"

save_data = {
    'results': results,
    'mwpm_results': {str(d): {str(p): v for p, v in ler_dict.items()}
                     for d, ler_dict in mwpm_results.items()},
    'config': {
        'test_distances': TEST_DISTANCES,
        'p_values': P_VALUES,
        'samples_per_p': SAMPLES_PER_P,
    },
    'timestamp': datetime.now().isoformat(),
}

with open(results_path, 'w') as f:
    json.dump(save_data, f, indent=2)

print(f"Results saved to: {results_path}")

# Save summary CSV
summary_path = RESULTS_DIR / "extrapolation_summary.csv"
df_summary.to_csv(summary_path, index=False)
print(f"Summary saved to: {summary_path}")

## Visualization

In [ ]:
# Plot accuracy by distance for each split
fig, axes = plt.subplots(1, len(TEST_DISTANCES), figsize=(5*len(TEST_DISTANCES), 5))

if len(TEST_DISTANCES) == 1:
    axes = [axes]

for idx, d in enumerate(TEST_DISTANCES):
    ax = axes[idx]

    # Plot each split
    for split_name in loaded_models.keys():
        split_d_results = df_results[(df_results['split_name'] == split_name) & (df_results['distance'] == d)]
        split_d_results = split_d_results.sort_values('p_value')
        ax.plot(split_d_results['p_value'] * 1000, split_d_results['accuracy'] * 100,
                'o-', label=split_name, markersize=4)

    # Plot MWPM baseline
    mwpm_p = sorted(mwpm_results[d].keys())
    mwpm_acc = [1 - mwpm_results[d][p] for p in mwpm_p]
    ax.plot([p*1000 for p in mwpm_p], [a*100 for a in mwpm_acc],
            'k--', label='MWPM', linewidth=2)

    ax.set_xlabel('Physical Error Rate (×10⁻³)')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title(f'd = {d}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()

plot_path = PLOTS_DIR / "extrapolation_accuracy.png"
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f"Plot saved to: {plot_path}")

plt.show()

In [ ]:
# Plot average accuracy by split across all distances
fig, ax = plt.subplots(figsize=(12, 6))

split_names = list(loaded_models.keys())
x = np.arange(len(split_names))
width = 0.2

for i, d in enumerate(TEST_DISTANCES):
    accs = []
    for split_name in split_names:
        split_d_results = df_results[(df_results['split_name'] == split_name) & (df_results['distance'] == d)]
        accs.append(split_d_results['accuracy'].mean() * 100)
    ax.bar(x + i*width, accs, width, label=f'd={d}')

ax.set_xlabel('Split Configuration')
ax.set_ylabel('Average Accuracy (%)')
ax.set_title('DeepSets Extrapolation Performance by Split and Distance')
ax.set_xticks(x + width)
ax.set_xticklabels(split_names, rotation=45, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()

plot_path = PLOTS_DIR / "extrapolation_by_split.png"
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
print(f"Plot saved to: {plot_path}")

plt.show()

In [ ]:
# Best performing split for each distance
print("\n" + "="*70)
print("BEST PERFORMING SPLITS")
print("="*70)

for d in TEST_DISTANCES:
    d_summary = df_summary[df_summary['Distance'] == d]
    best_idx = d_summary['DeepSets Acc'].idxmax()
    best = d_summary.loc[best_idx]
    print(f"\nd = {d}:")
    print(f"  Best split: {best['Split']}")
    print(f"  DeepSets accuracy: {best['DeepSets Acc']:.2f}%")
    print(f"  MWPM accuracy: {best['MWPM Acc']:.2f}%")
    print(f"  Improvement over MWPM: {best['Improvement']:+.2f}%")

## Comparison with GraphSAGE (if available)

In [ ]:
# Try to load GraphSAGE results for comparison
gsage_results_path = BASE_PATH / "gSAGE" / "extrapolation" / "results" / "extrapolation_results.json"

if gsage_results_path.exists():
    print("Loading GraphSAGE results for comparison...")
    with open(gsage_results_path, 'r') as f:
        gsage_data = json.load(f)

    # Create comparison table
    gsage_results = gsage_data['results']
    gsage_df = pd.DataFrame(gsage_results)

    comparison_data = []
    for split_name in loaded_models.keys():
        for d in TEST_DISTANCES:
            # DeepSets results
            ds_results = df_results[(df_results['split_name'] == split_name) & (df_results['distance'] == d)]
            ds_acc = ds_results['accuracy'].mean() * 100 if len(ds_results) > 0 else None

            # GraphSAGE results
            gs_results = gsage_df[(gsage_df['split_name'] == split_name) & (gsage_df['distance'] == d)]
            gs_acc = gs_results['accuracy'].mean() * 100 if len(gs_results) > 0 else None

            # MWPM baseline
            mwpm_acc = (1 - np.mean(list(mwpm_results[d].values()))) * 100

            if ds_acc is not None:
                comparison_data.append({
                    'Split': split_name,
                    'Distance': d,
                    'DeepSets': ds_acc,
                    'GraphSAGE': gs_acc,
                    'MWPM': mwpm_acc,
                    'DS vs GS': (ds_acc - gs_acc) if gs_acc else None,
                })

    df_comparison = pd.DataFrame(comparison_data)
    print("\n" + "="*80)
    print("DEEPSETS vs GRAPHSAGE COMPARISON")
    print("="*80)
    print(df_comparison.to_string(index=False))

    # Save comparison
    comparison_path = RESULTS_DIR / "deepsets_vs_gsage_comparison.csv"
    df_comparison.to_csv(comparison_path, index=False)
    print(f"\nComparison saved to: {comparison_path}")
else:
    print("GraphSAGE results not found. Run gSAGE/extrapolation/testing.ipynb to enable comparison.")

## Conclusions

After running this notebook, analyze:

1. **Extrapolation Performance:** How well do DeepSets models trained on d=3,5,7 generalize to d=9,11,13?

2. **Split Comparison:** Which training data split works best for extrapolation?
   - Does removing d=3 help or hurt?
   - Does more d=7 data improve extrapolation?

3. **Model Comparison:** How does DeepSets compare to GraphSAGE?
   - Does the simpler architecture (no graph structure) perform comparably?
   - Is the permutation-invariant aggregation sufficient for QEC?

4. **MWPM Baseline:** How do learned decoders compare to the classical MWPM decoder?
   - At which error rates do learned decoders excel?
   - Where does MWPM still dominate?